# Call Agent AI — Technical Documentation

This notebook documents the **meeting room booking assistant** built for the Cubo Itau.

It covers:
- The technology stack (LangGraph, LangChain, Groq, SQLAlchemy, SQLite, Pydantic).
- The architecture: **Hexagonal (Ports & Adapters) + Vertical Slice**.
- Cross-cutting infrastructure: the configuration **Singleton**, the **logger**, and the global exception decorator.
- The domain layer: entities, value objects and business rules.
- The application layer: use cases and the LangGraph tool-calling agent.
- The infrastructure layer: SQLAlchemy repositories and the manual DI container.

> The notebook runs against the actual project code "no mocks", using the SQLite database configured in `.env`.


## 1. Technology stack

| Concern | Technology |
|---|---|
| LLM provider | **Groq** (OpenAI-compatible endpoint `https://api.groq.com/openai/v1`) via `langchain-openai`'s `ChatOpenAI` |
| Agent orchestration / tool calling | **LangGraph** via LangChain's `create_agent` API (`langchain.agents.create_agent`) + `MemorySaver` checkpointer for per-user conversation memory |
| Domain modelling | Plain Python dataclasses / value objects, framework-free |
| Persistence | **SQLAlchemy** ORM (Python) over **SQLite** |
| Configuration | `pydantic-settings` for `.env` + a hand-rolled **Singleton** (`ConfigManager`) for `config.yaml` (business rules & prompts) |
| Logging | Custom daily-rotating file logger (`core/logging/logger.py`), console output disabled by default |
| Cross-cutting error handling | `@handle_service_exceptions` decorator that maps low-level exceptions to a controlled `RuntimeError` |
| Auth | Two hardcoded demo users (`User1`, `User2`) verified against `config.yaml` |

The LLM never performs date/time or business-rule math itself: the system prompt (also stored in `config.yaml`) instructs it to always delegate to tools, and the tools call the domain/use-case layer, which is the single source of truth for business rules.


## 2. Architecture — Hexagonal + Vertical Slice

Two organizing principles are combined:

**Vertical slice** — the codebase is split into independent features/bounded contexts, each owning its full stack:
```
auth/              # login of the two fixed demo users
calling_agent/      # room booking assistant (the actual feature)
core/               # cross-cutting: config, logging, global exception handling
utils/              # shared technical utilities: DB session/models, timestamps
```

**Hexagonal (Ports & Adapters)** — inside every slice, dependencies always point *inward* to the domain:
```
<slice>/
  domain/            # entities, value objects, business rules, domain exceptions,
                      # and REPOSITORY PORTS (abstract interfaces) — zero framework imports
  application/        # use cases (orchestrate domain + ports) and, for calling_agent,
                      # the LangGraph agent + tools (the "driving" adapter side)
  infrastructure/      # ADAPTERS: SQLAlchemy repositories, the LLM client, the DI container
```

Concretely, for `calling_agent`:
- `domain/repositories.py` defines `BookingRepository` / `RoomRepository` as `ABC` **ports**.
- `infrastructure/persistence/sqlalchemy_repositories.py` implements those ports as **adapters** using SQLAlchemy.
- `application/use_cases/*` depend only on the ports (never on SQLAlchemy directly) — this is what makes the domain persistence-agnostic and easily testable.
- `application/agent/tools.py` is the *driving* adapter: it translates LLM tool calls into use-case calls.

This is why domain-level exceptions and repository *interfaces* correctly live under `domain/` — only interfaces, not implementations.


## 3. Notebook setup

Add the project root to `sys.path` so the notebook can import the real project packages (run this notebook from `docs/` with the project's `.venv` as kernel).


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root added to sys.path: {PROJECT_ROOT}")


## 4. Configuration Singleton — `ConfigManager`

`core/config/config_manager.py` combines two sources of configuration:
- **`.env`** (secrets/environment, e.g. DB engine, Groq API key/URL/model) → loaded once via `pydantic-settings`' `BaseSettings` (`_env`), which validates required fields at import time and fails fast if something is missing.
- **`config.yaml`** (business config: room list, business hours, slot size, max duration, the agent system prompt, the two demo users) → loaded lazily and cached in memory.

The Singleton is implemented by overriding `__new__` so only one instance of `ConfigManager` ever exists per process:

```python
class ConfigManager:
    _instance: "ConfigManager | None" = None
    _config: dict | None = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
```

All access happens through `classmethod`s (`ConfigManager.get(...)`, `ConfigManager.get_sqlite_db_path()`, ...), so callers never instantiate it directly — this guarantees the YAML file is parsed only once and every part of the app reads a single consistent snapshot of the configuration, including the DB session builder (`utils/database/session.py`) and the agent's system prompt.


In [ ]:
from core.config.config_manager import ConfigManager

# Business rules loaded straight from config.yaml
print("DB engine:", ConfigManager.get_db_engine())
print("Rooms:", ConfigManager.get("call_agent.rooms"))
print("Business hours:", ConfigManager.get("call_agent.business_hours"))
print("Slot minutes:", ConfigManager.get("call_agent.slot_minutes"))
print("Max duration (min):", ConfigManager.get("call_agent.max_duration_minutes"))

# Proving the Singleton: two "instances" are actually the same object
print("Singleton check:", ConfigManager() is ConfigManager())


## 5. Logging — daily rotating file logger

`core/logging/logger.py` provides `setup_logger(name)`, used everywhere instead of `print`:

- Writes go to **rotating daily files**: `logs/info_<date>.log` (DEBUG..WARNING) and `logs/error_<date>.log` (ERROR+), each keeping the last 7 days (`_DailyFileHandler` purges older files automatically).
- Console (`stdout`) output is **off by default** and only enabled if `LOG_STDOUT=true` in `.env` — matching the requirement of never dumping things to the console.
- `LOG_ENABLED=false` swaps in a `NullHandler`, fully disabling logging without touching call sites.
- Every module gets its own named logger (`setup_logger(__name__)`), and handlers are attached only once per logger instance (`if logger.handlers: return logger`), so repeated imports don't duplicate log lines.

## 6. Global exception handling — `@handle_service_exceptions`

`core/exceptions/exceptions_decorators.py` exposes a decorator that wraps service-level functions and:
- Logs the original exception (with `logger.exception`/`logger.error`) including which function failed.
- Translates infra-level exceptions (`HTTPError`, `URLError`, `ConnectionError`, `TimeoutError`, `FileNotFoundError`, `ValueError`) into a single controlled `RuntimeError`, so the LangGraph tool layer only ever needs to catch two exception families: **domain exceptions** (`BookingDomainError` subclasses, expected/business errors) and **`RuntimeError`** (unexpected/infra errors) — see `application/agent/tools.py`.
- Re-raises `SystemExit`/`KeyboardInterrupt`/`GeneratorExit` untouched so the process can still shut down cleanly.


In [ ]:
from core.logging.logger import setup_logger

demo_logger = setup_logger("docs.demo")
demo_logger.info("Example info log line written from the notebook (check logs/info_<date>.log)")
demo_logger.error("Example error log line written from the notebook (check logs/error_<date>.log)")
print("Log lines written. LOG_STDOUT is disabled by default, so nothing prints to console except this line.")


## 7. Domain layer — entities, value objects and business rules

`calling_agent/domain/`:
- `entities.py`: `Room` (code + capacity) and `Booking` (id, room_code, user_id, title, attendees, start/end time).
- `value_objects.py`: `TimeSlot`, which knows how to check block-alignment (30-minute grid) and overlap against another slot — pure, framework-free logic.
- `rules.py`: `BookingRules`, the single place where every hard business rule from the challenge is enforced:
  1. `validate_slot` — end after start, aligned to `slot_minutes` blocks, duration ≤ `max_duration_minutes` (180), inside business hours.
  2. `validate_capacity` — attendees > 0 and ≤ room capacity.
  3. `validate_no_overlap` — queries existing bookings for that room/time window via the `BookingRepository` port and rejects overlapping ones.
- `exceptions.py`: `BookingDomainError` and its subclasses (`InvalidTimeSlotError`, `DurationExceededError`, `OutsideBusinessHoursError`, `CapacityExceededError`, `OverlapError`, ...) — these are business-rule violations, never infrastructure errors.

Below, the rules are exercised directly (no DB, no LLM) to show the business logic in isolation.


In [ ]:
from datetime import datetime, time

from calling_agent.domain.entities import Room
from calling_agent.domain.exceptions import BookingDomainError
from calling_agent.domain.rules import BookingRules
from calling_agent.domain.value_objects import TimeSlot

rules = BookingRules(
    slot_minutes=30,
    max_duration_minutes=180,
    business_start=time(8, 0),
    business_end=time(20, 0),
)
room_a = Room(code="A", capacity=4)

# 1) A valid 1-hour slot passes silently
valid_slot = TimeSlot(datetime(2026, 8, 4, 9, 0), datetime(2026, 8, 4, 10, 0))
rules.validate_slot(valid_slot)
rules.validate_capacity(room_a, attendees=3)
print("Valid booking accepted.")

# 2) A booking longer than 3 hours is rejected
try:
    too_long = TimeSlot(datetime(2026, 8, 4, 9, 0), datetime(2026, 8, 4, 13, 0))
    rules.validate_slot(too_long)
except BookingDomainError as e:
    print("Rejected (duration):", e)

# 3) A booking exceeding room capacity is rejected
try:
    rules.validate_capacity(room_a, attendees=10)
except BookingDomainError as e:
    print("Rejected (capacity):", e)

# 4) A booking not aligned to the 30-minute grid is rejected
try:
    misaligned = TimeSlot(datetime(2026, 8, 4, 9, 5), datetime(2026, 8, 4, 10, 0))
    rules.validate_slot(misaligned)
except BookingDomainError as e:
    print("Rejected (grid alignment):", e)


## 8. Application & Infrastructure — use cases, repositories and the DI container

- `application/use_cases/*` (`CreateBookingUseCase`, `CancelBookingUseCase`, `CheckAvailabilityUseCase`, `ListBookingsUseCase`): each is a small orchestrator that calls `BookingRules` and the repository **ports**. They know nothing about SQLAlchemy, SQLite, or LangChain.
- `infrastructure/persistence/sqlalchemy_repositories.py`: `SqlAlchemyBookingRepository` / `SqlAlchemyRoomRepository` implement the ports using SQLAlchemy sessions built from `utils/database/session.py` (engine path comes from `ConfigManager.get_sqlite_db_path()`).
- `infrastructure/container.py`: `BookingContainer` is a manual (no framework) dependency-injection container. It builds the repositories, reads booking-rule parameters from `config.yaml` via `ConfigManager`, builds `BookingRules`, and wires every use case — this is the composition root for the `calling_agent` slice.

Below we build a container and create a real booking against the SQLite database.


In [ ]:
from datetime import datetime

from calling_agent.application.dto import CancelBookingRequest, CreateBookingRequest
from calling_agent.domain.exceptions import BookingDomainError
from calling_agent.infrastructure.container import BookingContainer
from utils.database.init_db import init_db

init_db()  # ensures tables + the 5 fixed rooms (A-E) exist
container = BookingContainer()

request = CreateBookingRequest(
    user_id="User1",
    room_code="B",
    title="Docs demo standup",
    attendees=4,
    start_time=datetime(2026, 8, 5, 10, 0),
    end_time=datetime(2026, 8, 5, 10, 30),
)
booking = container.create_booking.execute(request)
print("Created booking:", booking)

# Trying to double-book the same slot fails with a domain error
try:
    container.create_booking.execute(request)
except BookingDomainError as e:
    print("Overlap correctly rejected:", e)

# Listing this user's bookings
for b in container.list_bookings.execute("User1"):
    print("User1 booking:", b)

# Clean up so re-running this notebook stays idempotent
container.cancel_booking.execute(CancelBookingRequest(user_id="User1", booking_id=booking.id))
print(f"Booking {booking.id} cancelled (cleanup).")


## 9. LangGraph agent and tool calling

- `infrastructure/llm/llm_client.py`: `build_chat_model()` returns a `ChatOpenAI` pointed at Groq's OpenAI-compatible endpoint, configured entirely through `ConfigManager` (model, base URL, API key, temperature, `max_tokens`, `max_retries` all come from `.env`/`config.yaml`).
- `application/agent/tools.py`: `build_agent_tools(user_id, container)` builds four `StructuredTool`s bound to the **currently authenticated user** (never passed by the LLM, so a user can never act on someone else's bookings):
  - `create_booking`, `cancel_booking`, `check_availability`, `list_my_bookings`.
  Each tool parses/validates its `pydantic` args schema, calls the corresponding use case, and converts `BookingDomainError` into a friendly string (the LLM narrates it back to the user) while unexpected errors are logged and returned as a generic apology — this keeps the LLM from ever seeing raw stack traces.
- `application/agent/graph_builder.py`: `build_agent_for_user(user)` wires everything with LangChain's modern `create_agent(model, tools, system_prompt, checkpointer)` API (built on top of LangGraph under the hood): model + tools + system prompt (loaded from `config.yaml`'s `call_agent.system_prompt`, with the active user appended) + a `MemorySaver` checkpointer so each user's `thread_id` keeps its own conversation memory.
- `main.py`: the CLI entry point — logs in (`AuthService` + `StaticAuthRepository`), builds the agent for that user, and runs a simple REPL loop calling `agent.invoke(...)`.

The system prompt itself lives in `config.yaml` (not hardcoded in Python), so it can be tuned without touching code — this satisfies the "prompts in the YAML config" requirement.


In [ ]:
print(ConfigManager.get("call_agent.system_prompt"))


## 10. Auth and end-to-end example

`auth/` is its own vertical slice, also hexagonal:
- `domain/entities.py` (`User`), `domain/exceptions.py` (`InvalidCredentialsError`), `domain/repositories.py` (`AuthRepository` port).
- `infrastructure/static_auth_repository.py`: adapter that checks credentials against the two fixed users declared in `config.yaml` (`User1`/`User2`).
- `application/auth_service.py`: `AuthService.login(username, password)` — the use case the CLI calls.

Below, a full end-to-end run: log in as `User1`, then let the **real LangGraph agent + Groq LLM** create a booking purely from a natural-language instruction, exercising tool calling live.


In [ ]:
from langchain_core.messages import HumanMessage

from auth.application.auth_service import AuthService
from auth.infrastructure.static_auth_repository import StaticAuthRepository
from calling_agent.application.agent.graph_builder import build_agent_for_user

auth_service = AuthService(StaticAuthRepository())
user = auth_service.login("User1", "User1Pass!")
print("Logged in as:", user)

agent = build_agent_for_user(user)
config = {"configurable": {"thread_id": user.user_id}}

reply = agent.invoke(
    {"messages": [HumanMessage(content="Book room C tomorrow 2026-08-06 from 15:00 to 16:00 for 5 people, title 'Sprint Review'")]},
    config=config,
)
print("Assistant:", reply["messages"][-1].content)

reply2 = agent.invoke({"messages": [HumanMessage(content="What are my current bookings?")]}, config=config)
print("Assistant:", reply2["messages"][-1].content)


In [2]:
## 11. Summary

- **Architecture**: Vertical Slice (`auth`, `calling_agent`, `core`, `utils`) + Hexagonal per slice (`domain` ports/entities/rules → `application` use cases/agent → `infrastructure` adapters).
- **Business rules** (5 rooms A-E, 30-minute contiguous blocks, no overlaps, ≤180 minutes, mandatory title/attendees, per-user isolation) are all enforced in `calling_agent/domain`, never in the LLM or the tool layer.
- **LangGraph** provides the tool-calling ReAct agent; **Groq** is the LLM provider through the OpenAI-compatible API; prompts and business parameters live in `config.yaml`, not hardcoded.
- **Cross-cutting concerns** — the `ConfigManager` Singleton, the daily-rotating file logger, and the `@handle_service_exceptions` global-exception decorator — are shared across every slice, keeping infra concerns out of the domain.
- **SQLite + SQLAlchemy** persist rooms and bookings behind repository ports, keeping the domain and use cases fully persistence-agnostic and unit-testable without a database.
- Fast package management using **uv** 


Error: No connection selected.